# Weekly Sector Rotation — raw price data pull

Pulls daily OHLCV (open, high, low, close, adjusted close, volume) for the
11 sector SPDR-style ETFs plus `VTI` (total-market benchmark, needed for
relative-return / benchmark-context calculations downstream).

Real network data only, via `yfinance` — no synthetic substitution. Raw
data is saved under `data/raw/weekly_sector_rotation/prices/` as one CSV
per symbol so this pull doesn't need to be re-run on every notebook
restart.

In [1]:
from pathlib import Path

import pandas as pd
import yfinance as yf

SECTOR_ETFS = [
    "VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE",
]
BENCHMARK_ETF = "VTI"  # total-market benchmark; needed for relative returns / benchmark context

SYMBOLS = SECTOR_ETFS + [BENCHMARK_ETF]

REPO_ROOT = Path.cwd().resolve().parents[3]
RAW_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
RAW_ROOT.mkdir(parents=True, exist_ok=True)
REPO_ROOT, RAW_ROOT

(PosixPath('/Users/gavinhussey/Downloads/AtlasQuant'),
 PosixPath('/Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices'))

In [2]:
def fetch_ohlcv(symbol: str) -> pd.DataFrame:
    """Fetch full daily OHLCV history for one symbol via yfinance.

    ``auto_adjust=False`` is required to get both the raw ``Close`` and a
    distinct ``Adj Close`` column back — with ``auto_adjust=True`` (the
    yfinance default) the ``Close`` column is silently replaced by the
    adjusted value and no unadjusted price survives.
    """
    ticker = yf.Ticker(symbol)
    history = ticker.history(period="max", auto_adjust=False, actions=False)
    if history.empty:
        raise RuntimeError(f"yfinance returned no data for {symbol!r}")

    history = history.reset_index()
    history["symbol"] = symbol
    history = history.rename(
        columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adjusted_close",
            "Volume": "volume",
        }
    )
    history["date"] = pd.to_datetime(history["date"]).dt.tz_localize(None).dt.date
    return history[["symbol", "date", "open", "high", "low", "close", "adjusted_close", "volume"]]


price_frames: dict[str, pd.DataFrame] = {}
for symbol in SYMBOLS:
    df = fetch_ohlcv(symbol)
    price_frames[symbol] = df
    out_path = RAW_ROOT / f"{symbol}.csv"
    df.to_csv(out_path, index=False)
    print(f"{symbol}: {len(df):,} rows, {df['date'].min()} -> {df['date'].max()} -> {out_path}")

VGT: 5,669 rows, 2004-01-30 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VGT.csv


VHT: 5,668 rows, 2004-01-30 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VHT.csv


VCR: 5,669 rows, 2004-01-30 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VCR.csv


VOX: 5,502 rows, 2004-09-29 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VOX.csv


VFH: 5,666 rows, 2004-01-30 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VFH.csv


...

VPU: 5,666 rows, 2004-01-30 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VPU.csv


VAW: 5,666 rows, 2004-01-30 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VAW.csv


VNQ: 5,501 rows, 2004-09-29 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VNQ.csv


VDE: 5,502 rows, 2004-09-29 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VDE.csv


VTI: 6,326 rows, 2001-06-15 -> 2026-08-12 -> /Users/gavinhussey/Downloads/AtlasQuant/data/raw/weekly_sector_rotation/prices/VTI.csv


In [3]:
prices = pd.concat(price_frames.values(), ignore_index=True).sort_values(["symbol", "date"])
prices.groupby("symbol").agg(
    rows=("date", "size"), first_date=("date", "min"), last_date=("date", "max")
)

,rows,first_date,last_date
symbol,,,
VAW,5666,2004-01-30,2026-08-12
VCR,5669,2004-01-30,2026-08-12
VDC,5669,2004-01-30,2026-08-12
VDE,5502,2004-09-29,2026-08-12
VFH,5666,2004-01-30,2026-08-12
VGT,5669,2004-01-30,2026-08-12
VHT,5668,2004-01-30,2026-08-12
VIS,5502,2004-09-29,2026-08-12
VNQ,5501,2004-09-29,2026-08-12
